# Week 10: Lecture 2
## Transposition
- The process of converting the orientation of a table is known as **transposition**

In [ ]:
import pandas as pd

df = pd.DataFrame(
    {
        "student": ["A", "B", "C"],
        "iPDI": [75, 65, 82],
        "PDI1": [85, 62, 75],
        "PDI2": [78, 68, 62],
    }
)

In [ ]:
df

- pandas provides the `.T` accessor or `transpose()` method to change the orientation of the data
- Doesn't change the structure or relationships in the data
- Rotates the data 90 degrees, columns become indices
- Useful for certain kinds of statistical analysis and visualisation

In [ ]:
df.T

## Long vs Wide Data
- Wide and long data formats represent the same information in fundamentally different ways
  - **Wide format**: each row represents a single entity
  - **Long format**: stacks this data vertically using categorical columns
- Wide data formats are preferable for presentation and summary tables
- Long format data is typically considered *tidy* because it properly separates variables into columns
  - Preferable for statistical analysis and visualisation
- The [Tidy Data](https://onesearch.library.northeastern.edu/permalink/01NEU_INST/1jo8mhm/cdi_doaj_primary_oai_doaj_org_article_6ab280a7154547029e9a3ee9d8b6c767) structure proposed by Wickham defines a standard way of structuring a dataset

### Tidy Data

  1. Each variable forms a column
  2. Each observation forms a row
  3. Each distinct type of observational unit should have its own table

## Melting and Pivoting
### Melting and Pivoting
- We can use the `melt()` method to [transform "wide" data into "long" format](https://pandas.pydata.org/docs/user_guide/reshaping.html#melt-and-wide-to-long)
- Unlike transposition, melting changes the shape and structure of the data
- The `id_vars` attribute is used to specify the columns you want to keep
- The `var_name` attribute contains the name of the old column names as values

In [ ]:
long = df.melt(id_vars="student", var_name="course", value_name="score")
long

- We can turn this back into a wide format using the [pivot method](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.pivot.html)
- Pivoting back is often necesary to produce visualisations where a wide format is preferred

In [ ]:
long.pivot(index="student", columns="course", values="score")

## Melting Multiple Columns
- If there are multiple columns to melt, we can use `value_vars` and `id_vars` attributes
- Suppose we start with a table like this:

In [ ]:
df = pd.DataFrame(
    {
        "student": ["A", "B", "C"],
        "iPDI": [75, 65, 82],
        "PDI1": [85, 62, 75],
        "PDI2": [78, 68, 62],
        "major": ["Humanities", "CS", "Physics"],
    }
)

df

In [ ]:
df = df.melt(
    id_vars=["student", "major"],
    value_vars=["iPDI", "PDI1", "PDI2"],
    var_name="course",
    value_name="score",
)

df

- We can pivot this back to the original DataFrame as follows
- The [reset_index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html) removes the MultiIndex

In [ ]:
df.pivot(index=["student", "major"], columns="course", values="score").reset_index()

## Grouping
- Long data formats produced by melting are optimal for performing [grouping operations](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
- Grouping enables you to [slice datasets by values](https://pandas.pydata.org/docs/user_guide/groupby.html)
  - Calculate groupwise statistics and apply transformations to different subsets of data
- Group by consists of three operations: [split, apply, combine](https://www.jstatsoft.org/article/view/v040i01)
  - Data is *split* into groups based on keys
  - A function is *applied* to each group
  - The results of those functions are *combined* into a new result object

## GroupBy
- Grouped data creates a GroupBy object

In [ ]:
long.groupby("course")

- We can get the [indicies](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.core.groupby.DataFrameGroupBy.indices.html#pandas.core.groupby.DataFrameGroupBy.indices) of the groups inside that object using the `indices` attribute
- This returns a dictionary with keys for each group and values containing their corresponding row index

In [ ]:
df.groupby("course").indices

- We can use the indices to `get` particular groups with an `iloc`

In [ ]:
df.iloc[long.groupby("course").indices.get("PDI1")]

## Aggregating a GroupBy
- GroupBys are typically aggregated to create a computed dataframe for grouped data
- We can use a number of [built-in aggregation methods](https://pandas.pydata.org/docs/user_guide/groupby.html#built-in-aggregation-methods) which can be applied to all groups
- The size method is useful for getting the size of the groups

In [ ]:
df.groupby("major")["score"].size()

- Numerical columns can be used to perform statistical calculations

In [ ]:
df.groupby("course")["score"].max()

- nlargest and nsmallest are useful for getting sets of smallest and largest values

In [ ]:
df.groupby("course")["score"].nlargest(2)

- We can group data using two keys to create a hierarchical index
- This allows us to group by multiple categories
- Returns a MultiIndex with grouping by multiple columns

In [ ]:
df.groupby(["major", "course"])["score"].mean()

- This is currently a series with a single column of values, with two indices

In [ ]:
df.groupby(["major", "course"])["score"].mean()["CS"]

- We can [unstack](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.unstack.html) the Series so that it is pivoted into a wide dataset

In [ ]:
df.groupby(["major", "course"])["score"].mean().unstack()

- This makes it easier to plot charts for the different groups

In [ ]:
df.groupby(["major", "course"])["score"].mean().unstack().plot(kind="bar")

## Class Exercises
Load `photos.csv` and answer the following question without using `value_counts`
- How many photos were taken in each location?
- What is the most photographed subject?

- A quick way to create a year column in the `photos` dataset is by extracting the first four characters of the `Date` column
```python
photos["Date"].str[0:4]
```
  - Group by year and count how many photos were taken each year
  - Create a bar plot showing photos per year

- Create a visualization showing which locations were most photographed each year.
  - Hint: Group by both year and location

Load `workouts.csv` and answer the following questions:
For each exercise type, calculate:
- Average duration
- Average calories burned
- Average heart rate

- Calculate the "calories per minute" for each workout, then:
  - Find which exercise type is most efficient on average
  - Create a box plot showing the distribution of efficiency by exercise type